In [36]:
# Importing all required modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

# Importing the models and train_test_split
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score



In [37]:
# Load the dataset 
data = pd.read_csv('artDataset.csv')

In [38]:
# Explore data
# data.describe()
data.head()

,Unnamed: 0,price,artist,title,yearCreation,signed,condition,period,movement
0,0,28.500 USD,Tommaso Ottieri,Bayreuth Opera,2021,Signed on verso,This work is in excellent condition.,Contemporary,Baroque
1,1,3.000 USD,Pavel Tchelitchew,Drawings of the Opera,First Half 20th Century,Signed and titled,Not examined out of frame.No obvious signs of ...,Post-War,Surrealism
2,2,5.000 USD,Leo Gabin,Two on Sidewalk,2016,"Signed, titled and dated on verso",This work is in excellent condition.,Contemporary,Abstract
3,3,5.000 USD,Matthias Dornfeld,Blumenszene,2010,"Signed, titled and dated on the reverse with t...",This work is in excellent condition.There is m...,Contemporary,Abstract
4,4,2.500 USD,Alexis Marguerite Teplin,Feverish Embarkation,2001,Signed on verso,This work is in excellent condition.,Contemporary,Abstract


In [54]:
# Check for missing values
data.isnull().sum()

Unnamed: 0      0
price           0
artist          0
title           0
yearCreation    0
signed          0
condition       0
period          0
movement        0
dtype: int64

**Data Transformation**: transformations to the data to ensure consistency.

In [40]:
# Keep only rows where 'yearCreation' is exactly 4 digits, remove any string text
data = data[data["yearCreation"].astype(str).str.match(r"^\d{4}$")]

# Convert to integer
data["yearCreation"] = data["yearCreation"].astype(int)

In [41]:
# Ensure price is an int or float 
# Convert all values to strings
data["price"] = data["price"].astype(str)

# Remove " USD" and any "."
data["price"] = data["price"].str.replace(" USD", "", regex=False).str.replace(".", "", regex=False)

# Convert to numeric
data["price"] = pd.to_numeric(data["price"], errors="coerce")

# Drop any rows where conversion failed
data = data.dropna(subset=["price"])

**Initialize the Benchmark Model**: CatBoost

In [42]:
# Define parameters like iterations, depth, learning rate etc)
benchmark = CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

**Assign X and y values** for the benchmark model

In [43]:
# For this project, the benchmark will use a subset of features
X_benchmark = data[["yearCreation", "movement"]]
y_benchmark = data["price"]

# Split the dataset for training and testing, with a random state to ensure consistency 
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_benchmark, y_benchmark, train_size=0.8, random_state=26)


**Identify categorical features** and assign to a variable

In [44]:
# In this instance, 'movement' is the only categorical feature that is being included in the benchmark
categorical_features = ["movement"]


**Model Training and Prediction**

In [45]:
# Fit the benchmark model, including the categorical features
benchmark.fit(X_train_b, y_train_b, cat_features=categorical_features)

0:	learn: 12956.3699991	total: 50.9ms	remaining: 5.03s
99:	learn: 10730.3413263	total: 3.38s	remaining: 0us


In [46]:
# Predict prices based on the year using test data
benchmark_prediction = benchmark.predict(X_test_b)
print(benchmark_prediction)

[17544.60278813  1943.26092619  1993.60267089  3533.73814006
  1514.23710353  3051.89053848  1757.5203143   5874.08180714
 16592.8321438  10668.70104562  4714.78788678  1908.67347578
  9690.84063844  1677.291314    9426.58098663  3741.06407168
  1677.291314    3229.05280228  4317.64381775  4834.4888665
 10295.55700816  4436.01860045  1514.23710353 14308.58288152
  6978.43147235 10197.60657288  1677.291314    3418.11711337
  9625.70465807  3772.11979571  3934.26381168  3822.80088118
  2064.22642487  6397.8228642   1677.291314    4134.23811062
  1677.291314    7382.53758581  3533.73814006  9159.28791196
 10862.73579232  6162.15736414  9109.13395226  3028.42639252
  2064.22642487 11518.87939259 10238.90475058  6431.56104907
  4184.57985532  5953.70329038  3073.51869661  3916.22794235
  9530.81597188 10471.78520196 17159.38571649  2683.23649535
  6145.49372785  1828.44447547  7382.53758581  7769.15048467
  2040.101109    8943.25229064  2036.74193275  5874.08180714
  1993.60267089  1853.774

**Evaluate the benchmark model**: Metrics include 
- R2 Score - this explains how much variation in the target variable is explained by the model.
- Mean Absolute Error - measures the average size of the prediction errors

In [47]:
# Evaluate the benchmark model using R2 and MAE
print("R2:", r2_score(y_test_b, benchmark_prediction))
print("MAE:", mean_absolute_error(y_test_b, benchmark_prediction))

R2: 0.1422346505224652
MAE: 5063.975793354287


**Primary Model**

**Initialize the primary model**: CatBoost 

In [48]:
# Define parameters like iterations, depth, learning rate etc)
cat_boost = CatBoostRegressor(
    iterations=1000,
    depth=10,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
    )

**Assign X and y** values - Here, all features are included.

In [49]:
# Assign the X (data) and y (target) labels 
X = data.drop(columns=["price"])
y = data["price"]

# Split the dataset for training and testing
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, random_state=2025)

**Define categorical features** and assign to a variable

In [50]:
# All non-numeric features are identified here as categorical features
categorical_features = ["artist", "title", "signed", "condition", "period", "movement"]


**Model Training and Prediction**

In [51]:
# Fit the model, including the categorical features
cat_boost.fit(X_train, y_train, cat_features=categorical_features)

0:	learn: 13342.0945460	total: 185ms	remaining: 3m 5s
100:	learn: 7661.1701463	total: 9.04s	remaining: 1m 20s
200:	learn: 4724.0065969	total: 19s	remaining: 1m 15s
300:	learn: 3163.8483106	total: 27.9s	remaining: 1m 4s
400:	learn: 2173.5197114	total: 39.3s	remaining: 58.6s
500:	learn: 1573.1950534	total: 54.1s	remaining: 53.9s
600:	learn: 1187.3047765	total: 1m 20s	remaining: 53.3s
700:	learn: 916.5452399	total: 1m 41s	remaining: 43.3s
800:	learn: 745.1194683	total: 1m 52s	remaining: 27.9s
900:	learn: 619.6675375	total: 2m 2s	remaining: 13.4s
999:	learn: 520.0948804	total: 2m 10s	remaining: 0us


In [52]:
# Model prediction using test data 
prediction = cat_boost.predict(X_test)

In [55]:
# Evaluate the benchmark model using R2 and MAE
print("R2:", r2_score(y_test, prediction))
print("MAE:", mean_absolute_error(y_test, prediction))

R2: -0.0509212947482387
MAE: 3693.2598680948486
